In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = "../data/clean/daily_stock_optimal_bucket_two_stage_modeling.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.columns.tolist())
df.head()

Dataset shape: (43766, 139)
['date', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'dividend', 'split_coeff', 'fiscalDateEnding', 'totalRevenue', 'grossProfit', 'operatingIncome', 'operatingExpenses', 'ebit', 'ebitda', 'totalAssets', 'totalCurrentAssets', 'cashAndCashEquivalentsAtCarryingValue', 'totalLiabilities', 'totalCurrentLiabilities', 'longTermDebt', 'totalShareholderEquity', 'commonStockSharesOutstanding', 'operatingCashflow', 'capitalExpenditures', 'cashflowFromInvestment', 'cashflowFromFinancing', 'dividendPayout', 'netIncome', 'MarketCapitalization', 'PERatio', 'BookValue', 'DividendYield', 'EPS', 'RevenueTTM', 'ProfitMargin', 'OperatingMarginTTM', 'ReturnOnAssetsTTM', 'ReturnOnEquityTTM', 'Beta', '52WeekHigh', '52WeekLow', '50DayMovingAverage', '200DayMovingAverage', 'SharesOutstanding', 'return_1d', 'log_return', 'momentum_1m', 'momentum_3m', 'momentum_6m', 'ma_20', 'ma_50', 'ma_200', 'price_vs_ma50', 'price_vs_ma200', 'vol_20', 'vol_60', 'parkinson_dail

,date,symbol,open,high,low,close,adj_close,volume,dividend,split_coeff,...,d_premium_mean_90,d_theta_mean_30,d_theta_mean_60,d_theta_mean_90,d_vega_mean_30,d_vega_mean_60,d_vega_mean_90,optimal_bucket,moneyness_target,dte_target
0,2006-01-03,AAPL,72.332,74.75,72.25,74.75,2.239667,28829800,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OTM5_30,OTM5,30
1,2006-01-04,AAPL,75.130,75.98,74.50,74.97,2.246259,22128700,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OTM5_30,OTM5,30
2,2006-01-05,AAPL,74.830,74.90,73.75,74.38,2.228582,16050800,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OTM5_30,OTM5,30
3,2006-01-06,AAPL,75.240,76.70,74.55,76.30,2.286109,25159200,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OTM5_30,OTM5,30
4,2006-01-09,AAPL,76.730,77.20,75.74,76.05,2.278618,24108600,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OTM5_30,OTM5,30


In [2]:
# Basic Checks

df["date"] = pd.to_datetime(df["date"])

print("Duplicate symbol-date rows:", df.duplicated(subset=["symbol", "date"]).sum())

null_summary = df.isnull().sum().sort_values(ascending=False)
print("\nColumns with nulls:")
print(null_summary[null_summary > 0])

Duplicate symbol-date rows: 0

Columns with nulls:
d_bucket_count_60         42552
d_delta_mean_60           42552
d_iv_mean_60              42552
d_oi_sum_60               42552
d_vega_mean_60            42552
d_theta_mean_60           42552
d_premium_mean_60         42552
m_vega_mean_ATM           42313
m_theta_mean_ATM          42313
m_premium_mean_ATM        42313
m_oi_sum_ATM              42313
m_iv_mean_ATM             42313
m_delta_mean_ATM          42313
m_bucket_count_ATM        42313
m_iv_mean_OTM5            42292
m_delta_mean_OTM5         42292
m_theta_mean_OTM5         42292
m_bucket_count_OTM5       42292
m_vega_mean_OTM5          42292
m_oi_sum_OTM5             42292
m_premium_mean_OTM5       42292
d_delta_mean_30           42284
d_theta_mean_30           42284
d_bucket_count_30         42284
d_iv_mean_30              42284
d_oi_sum_30               42284
d_vega_mean_30            42284
d_premium_mean_30         42284
d_delta_mean_90           42271
d_iv_mean_90         

In [3]:
# Create two-stage targets
df["moneyness_target"] = df["optimal_bucket"].str.split("_").str[0]
df["dte_target"] = df["optimal_bucket"].str.split("_").str[1]

print("Moneyness distribution:")
print(df["moneyness_target"].value_counts())

print("\nDTE distribution:")
print(df["dte_target"].value_counts())

Moneyness distribution:
moneyness_target
ATM      30210
OTM10     8047
OTM5      5509
Name: count, dtype: int64

DTE distribution:
dte_target
90    21502
30    12879
60     9385
Name: count, dtype: int64


In [4]:
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

In [ ]:
# one-hot encode symbol
# 
df_encoded = pd.get_dummies(
    df,
    columns=["symbol"],
    prefix="symbol"
)

symbol_cols = [col for col in df_encoded.columns if col.startswith("symbol_")]

df_encoded[symbol_cols] = df_encoded[symbol_cols].astype(int)

print("Symbol columns:", symbol_cols)

Symbol columns: ['symbol_AAPL', 'symbol_AMZN', 'symbol_AVGO', 'symbol_GOOG', 'symbol_GOOGL', 'symbol_META', 'symbol_MSFT', 'symbol_NVDA', 'symbol_TSLA', 'symbol_WMT']


In [6]:
# Fiscal date features
df_encoded["fiscalDateEnding"] = pd.to_datetime(df_encoded["fiscalDateEnding"])

df_encoded["days_since_fiscal_report"] = (
    df_encoded["date"] - df_encoded["fiscalDateEnding"]
).dt.days

df_encoded["fiscal_quarter"] = df_encoded["fiscalDateEnding"].dt.quarter

df_encoded = df_encoded.drop(columns=["fiscalDateEnding"])

In [7]:
# Encode targets (two-stage)

from sklearn.preprocessing import LabelEncoder

moneyness_encoder = LabelEncoder()
dte_encoder = LabelEncoder()

df_encoded["moneyness_encoded"] = moneyness_encoder.fit_transform(df_encoded["moneyness_target"])
df_encoded["dte_encoded"] = dte_encoder.fit_transform(df_encoded["dte_target"])

print("Moneyness mapping:")
print(dict(zip(moneyness_encoder.classes_, moneyness_encoder.transform(moneyness_encoder.classes_))))

print("\nDTE mapping:")
print(dict(zip(dte_encoder.classes_, dte_encoder.transform(dte_encoder.classes_))))

Moneyness mapping:
{'ATM': np.int64(0), 'OTM10': np.int64(1), 'OTM5': np.int64(2)}

DTE mapping:
{'30': np.int64(0), '60': np.int64(1), '90': np.int64(2)}


In [8]:
# drop targets + leakage features
exclude_cols = [
    "date",
    "optimal_bucket",
    "moneyness_target",
    "dte_target",
    "moneyness_encoded",
    "dte_encoded"
]

leakage_cols = [
    "covered_call_return",
    "realized_vol_cycle",
    "rf_cycle",
    "sharpe_proxy"
]

exclude_cols += [c for c in leakage_cols if c in df_encoded.columns]

In [9]:
# Build X safely
feature_df = df_encoded.drop(columns=[c for c in exclude_cols if c in df_encoded.columns]).copy()

# keep only numeric
feature_df = feature_df.select_dtypes(include=["number"]).copy()

print("Feature dataset shape:", feature_df.shape)
print(feature_df.columns.tolist())

Feature dataset shape: (43766, 145)
['open', 'high', 'low', 'close', 'adj_close', 'volume', 'dividend', 'split_coeff', 'totalRevenue', 'grossProfit', 'operatingIncome', 'operatingExpenses', 'ebit', 'ebitda', 'totalAssets', 'totalCurrentAssets', 'cashAndCashEquivalentsAtCarryingValue', 'totalLiabilities', 'totalCurrentLiabilities', 'longTermDebt', 'totalShareholderEquity', 'commonStockSharesOutstanding', 'operatingCashflow', 'capitalExpenditures', 'cashflowFromInvestment', 'cashflowFromFinancing', 'dividendPayout', 'netIncome', 'MarketCapitalization', 'PERatio', 'BookValue', 'DividendYield', 'EPS', 'RevenueTTM', 'ProfitMargin', 'OperatingMarginTTM', 'ReturnOnAssetsTTM', 'ReturnOnEquityTTM', 'Beta', '52WeekHigh', '52WeekLow', '50DayMovingAverage', '200DayMovingAverage', 'SharesOutstanding', 'return_1d', 'log_return', 'momentum_1m', 'momentum_3m', 'momentum_6m', 'ma_20', 'ma_50', 'ma_200', 'price_vs_ma50', 'price_vs_ma200', 'vol_20', 'vol_60', 'parkinson_daily', 'parkinson_vol_20', 'intra

In [10]:
# Time-based split
train_df = df_encoded[df_encoded["date"] < "2022-01-01"].copy()

val_df = df_encoded[
    (df_encoded["date"] >= "2022-01-01") &
    (df_encoded["date"] < "2024-01-01")
].copy()

test_df = df_encoded[df_encoded["date"] >= "2024-01-01"].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (33736, 151)
Val: (5010, 151)
Test: (5020, 151)


In [11]:
feature_cols = [c for c in feature_df.columns if c in df_encoded.columns]

X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

# targets
y_train_m = train_df["moneyness_encoded"]
y_val_m   = val_df["moneyness_encoded"]
y_test_m  = test_df["moneyness_encoded"]

y_train_d = train_df["dte_encoded"]
y_val_d   = val_df["dte_encoded"]
y_test_d  = test_df["dte_encoded"]

print("X_train:", X_train.shape)
print("y_train_m:", y_train_m.shape)
print("y_train_d:", y_train_d.shape)

X_train: (33736, 145)
y_train_m: (33736,)
y_train_d: (33736,)


In [12]:
print("Moneyness train distribution:")
print(y_train_m.value_counts())

print("\nDTE train distribution:")
print(y_train_d.value_counts())

Moneyness train distribution:
moneyness_encoded
0    25746
2     4128
1     3862
Name: count, dtype: int64

DTE train distribution:
dte_encoded
2    14563
0    11299
1     7874
Name: count, dtype: int64


In [16]:
# Correlation filtering
corr_matrix = X_train.corr()

corr_threshold = 0.90

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper_triangle.columns
    if any(abs(upper_triangle[column]) > corr_threshold)
]

print("Dropping:", len(high_corr_features))

X_train = X_train.drop(columns=high_corr_features, errors="ignore")
X_val   = X_val.drop(columns=high_corr_features, errors="ignore")
X_test  = X_test.drop(columns=high_corr_features, errors="ignore")

Dropping: 0


In [17]:
# scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [18]:
# Save datasets

MODEL_DATA_PATH = Path("../model_datasets_two_stage")
MODEL_DATA_PATH.mkdir(parents=True, exist_ok=True)

# features
X_train.to_parquet(MODEL_DATA_PATH / "X_train.parquet", index=False)
X_val.to_parquet(MODEL_DATA_PATH / "X_val.parquet", index=False)
X_test.to_parquet(MODEL_DATA_PATH / "X_test.parquet", index=False)

# scaled
X_train_scaled.to_parquet(MODEL_DATA_PATH / "X_train_scaled.parquet", index=False)
X_val_scaled.to_parquet(MODEL_DATA_PATH / "X_val_scaled.parquet", index=False)
X_test_scaled.to_parquet(MODEL_DATA_PATH / "X_test_scaled.parquet", index=False)

# targets
y_train_m.to_frame("moneyness_encoded").to_parquet(MODEL_DATA_PATH / "y_train_m.parquet", index=False)
y_val_m.to_frame("moneyness_encoded").to_parquet(MODEL_DATA_PATH / "y_val_m.parquet", index=False)
y_test_m.to_frame("moneyness_encoded").to_parquet(MODEL_DATA_PATH / "y_test_m.parquet", index=False)

y_train_d.to_frame("dte_encoded").to_parquet(MODEL_DATA_PATH / "y_train_d.parquet", index=False)
y_val_d.to_frame("dte_encoded").to_parquet(MODEL_DATA_PATH / "y_val_d.parquet", index=False)
y_test_d.to_frame("dte_encoded").to_parquet(MODEL_DATA_PATH / "y_test_d.parquet", index=False)

print("Saved successfully")

Saved successfully
